# Provider & Practice Directory Update Pipeline

## Option C Hybrid Submission: production architecture plus working MVP

Provider directories decay continuously: clinicians move, practice names change, phone numbers drift, websites disagree, and some providers become inactive. This notebook tells the end-to-end story of a repeatable pipeline that detects those changes, validates them against trusted sources, assigns confidence, separates safe automation from human review, and preserves an audit trail.

**Submission thesis:** HealthLynked should not buy a one-time cleanup. It should implement a governed, AWS-ready data quality loop that can run continuously with bounded cost and explainable decisions.

## Evaluation Criteria: What Judges Should See

| Requirement | Where this notebook demonstrates it |
|---|---|
| Technical Architecture Proposal | Architecture, agents, AWS production path, cost controls, update workflow |
| Working Prototype | Local MVP runs on sample provider/practice data |
| Accuracy | Precision, recall, F1, false positives, false negatives |
| Scalability | Batchable AWS Step Functions / Lambda / ECS / S3 / RDS design |
| Cost Efficiency | Source-first strategy, gated LLM use, cost per correct update |
| Practicality | Lean-team roadmap and reproducible scripts |
| Explainability | Candidate rows include old/new value, confidence, sources, URLs, and reasons |
| Data Quality | Normalization for phone, address, specialty, status, and matching keys |
| Source Reliability | Authority tiers, source conflict handling, source freshness controls |
| Human Review Design | Only risky, conflicting, or uncertain updates route to review |
| Audit Trail | Audit events, rollback plan, package manifest, source evidence |

AWS references used for the production design: [AWS Step Functions](https://docs.aws.amazon.com/step-functions/latest/dg/welcome.html) for workflow/state-machine orchestration and [Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/what-is-bedrock.html) for gated foundation-model extraction when deterministic parsing is insufficient.

## Whitepaper-Informed Design

The architecture follows the supplied agentic-engineering whitepaper notes:

- Build a **harness**, not one giant prompt: connectors, schemas, evals, review queues, dashboards, audit events, rollback plans, monitoring.
- Use **scoped agents and skills**: source retrieval, evidence normalization, identity resolution, conflict adjudication, scoring, review, audit, monitoring.
- Practice **progressive disclosure**: bulky source rules and authority tables live in docs/data/contracts, while code performs deterministic validation.
- Use **LLMs sparingly**: Amazon Bedrock is a gated fallback for messy official/practice pages, not the default data pipeline.
- Treat human review as a precision control, not a failure mode.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.run_best_pipeline import BEST_CFG
from src.data import load_dataset, build_candidate_updates, normalize_phone, normalize_address, normalize_value
from src.metrics import score_candidate_updates, per_field_breakdown, worst_provider_breakdown
from src.entity_resolution import potential_duplicate_pairs, provider_movement_signals
from src.inactive_detection import inactive_provider_candidates
from src.source_conflicts import source_conflict_diagnostics

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)
ROOT

## 1. Load Sample Provider/Practice Data

The competition did not provide conventional train/test labels, so the MVP uses a synthetic but healthcare-realistic provider directory benchmark. It includes current directory values, external source evidence, and proxy gold updates so the pipeline can be measured instead of merely described.

In [ ]:
providers, evidence, gold = load_dataset()

summary = pd.DataFrame([
    {'table': 'providers', 'rows': len(providers), 'columns': len(providers.columns)},
    {'table': 'evidence', 'rows': len(evidence), 'columns': len(evidence.columns)},
    {'table': 'gold_updates', 'rows': len(gold), 'columns': len(gold.columns)},
])
summary

In [ ]:
providers.head(8)

In [ ]:
evidence.head(10)

## 2. Run The MVP Pipeline

The MVP does the core work judges asked for:

1. Compare current records to external evidence.
2. Normalize values before comparison.
3. Require source support by field.
4. Score confidence.
5. Route to **auto-apply** or **human review**.
6. Emit reproducible CSV/JSON artifacts.

In [ ]:
candidates = build_candidate_updates(providers, evidence, BEST_CFG)
metrics = score_candidate_updates(candidates, gold)

pd.DataFrame([metrics]).T.rename(columns={0: 'value'})

In [ ]:
display_cols = [
    'provider_id', 'practice_id', 'field', 'old_value', 'proposed_value',
    'confidence', 'decision', 'distinct_sources', 'sources', 'review_reason_code', 'url'
]
candidates[[c for c in display_cols if c in candidates.columns]].head(12)

## 3. Accuracy: Correct Updates Without Reckless Mutation

The key metric is not just recall. In a healthcare directory, incorrect auto-updates are expensive and trust-eroding. This prototype therefore separates **candidate discovery** from **safe mutation**. The current configuration keeps auto-apply precision at `1.0` in the proxy benchmark.

In [ ]:
per_field_breakdown(candidates, gold)[['field', 'precision', 'recall', 'f1', 'predicted_updates', 'gold_updates', 'auto_apply_precision']]

In [ ]:
worst_provider_breakdown(candidates, gold, limit=10)

## 4. Confidence Scoring Formula

The production formula is intentionally explainable:

```text
confidence =
  source_authority_score
  + source_agreement_bonus
  + freshness_bonus
  + identity_anchor_bonus
  + extraction_quality_bonus
  - field_risk_penalty
  - conflict_penalty
  - stale_source_penalty
```

In the MVP this appears as source weights, recency gates, field thresholds, minimum source counts, conflict penalties, and stricter auto-apply thresholds.

In [ ]:
json.dumps(BEST_CFG, indent=2)

## 5. Human Review Design

The review queue is not a junk drawer. It contains records where review is actually useful: low confidence, source conflict, insufficient source support, high-risk field, identity-level concern, or peer mismatch. This is how the system reduces manual labor while avoiding unsafe automation.

In [ ]:
auto_apply = candidates[candidates['decision'] == 'auto_apply'].copy()
review_queue = candidates[candidates['decision'] == 'review'].copy()

pd.DataFrame([
    {'queue': 'auto_apply', 'rows': len(auto_apply), 'mean_confidence': round(auto_apply['confidence'].mean(), 4)},
    {'queue': 'human_review', 'rows': len(review_queue), 'mean_confidence': round(review_queue['confidence'].mean(), 4)},
])

In [ ]:
review_queue[[c for c in display_cols if c in review_queue.columns]].head(12)

## 6. Data Quality Normalization

Normalization prevents noisy formatting differences from becoming false updates. The MVP demonstrates phone, address, specialty, and status normalization. The production design extends this to provider names, practice names, websites, and affiliations.

In [ ]:
normalization_examples = pd.DataFrame([
    {'field': 'phone', 'raw': '(555) 123-4567', 'normalized': normalize_phone('(555) 123-4567')},
    {'field': 'phone', 'raw': '+1 555.123.4567', 'normalized': normalize_phone('+1 555.123.4567')},
    {'field': 'address', 'raw': '100 Main St., Ste 4, FL', 'normalized': normalize_address('100 Main St., Ste 4, FL')},
    {'field': 'specialty', 'raw': 'Cardiovascular Disease', 'normalized': normalize_value('specialty', 'Cardiovascular Disease')},
    {'field': 'active/inactive', 'raw': 'Not Active', 'normalized': normalize_value('license_status', 'Not Active')},
])
normalization_examples

## 7. Source Reliability And Conflict Handling

Trusted sources are not equal for every field. NPPES is strong for identity/taxonomy, state boards are strong for license status, practice websites are strong for location/phone/roster, and business listings are only a guarded fallback for fresh phone/address evidence.

In [ ]:
conflicts = source_conflict_diagnostics(evidence)
conflicts.head(10)

## 8. Duplicate, Movement, Practice Matching, And Inactive Provider Signals

Bonus capabilities are represented as diagnostics and review queues. Identity-sensitive actions are review-first; the goal is to surface strong evidence, not silently merge or deactivate providers.

In [ ]:
duplicates = potential_duplicate_pairs(providers)
movement = provider_movement_signals(candidates)
inactive = inactive_provider_candidates(providers, evidence)

print('duplicate candidates:', len(duplicates))
print('movement candidates:', len(movement))
print('inactive candidates:', len(inactive))

movement.head(8)

In [ ]:
inactive.head(8)

## 9. Cost Efficiency

The cost strategy is source-first and review-aware:

- Cache public snapshots.
- Refresh risky/stale records first.
- Use deterministic parsers before LLMs.
- Gate Amazon Bedrock fallback to messy approved pages.
- Avoid manual review unless the confidence logic says review is needed.
- Measure cost per correct update, not just total spend.

In [ ]:
cost_rows = [{'source': k, 'estimated_cost_usd': v} for k, v in metrics['source_cost_breakdown_usd'].items()]
pd.DataFrame(cost_rows).sort_values('estimated_cost_usd', ascending=False)

## 10. Audit Trail And Safe Auto-Update Rules

Every candidate must be explainable: old value, proposed value, source set, URL, confidence, decision, and reason. The production system writes immutable audit events and rollback plans before updating the directory.

In [ ]:
audit_path = ROOT / 'submissions/exp0172/audit_events.jsonl'
if audit_path.exists():
    audit_preview = [json.loads(line) for line in audit_path.read_text().splitlines()[:5]]
    display(pd.DataFrame(audit_preview))
else:
    print('Audit fixture not found in this checkout.')

## 11. Technical Architecture Diagram

```mermaid
flowchart TD
    A[HealthLynked Provider and Practice Directory] --> B[Risk Scanner]
    B --> C{Refresh Needed?}
    C -->|No| N[No Change / Confirm Current Record]
    C -->|Yes| D[Trusted Source Connector Orchestrator]
    D --> S1[NPPES / NPI Registry]
    D --> S2[State Licensing Boards]
    D --> S3[CMS / Public Datasets]
    D --> S4[Practice Websites]
    D --> S5[Health System Directories]
    D --> S6[Guarded Business Listing Fallback]
    S1 --> E[Evidence Store]
    S2 --> E
    S3 --> E
    S4 --> E
    S5 --> E
    S6 --> E
    E --> F[Clean and Normalize Data]
    F --> G[Provider / Practice Matching]
    G --> H[Validation Logic]
    H --> I[Confidence Score]
    I --> J{Decision}
    J -->|Record confirmed| N
    J -->|High-confidence update| K[Safe Auto Update]
    J -->|Low confidence or conflict| L[Human Review]
    K --> M[Audit Log + Rollback Plan]
    L --> M
    N --> M
    M --> O[Update Directory + Feedback Loop]
    O --> B
    P[Amazon Bedrock LLM Fallback] -. gated only .-> F
    D -. messy approved page .-> P
```

## 12. AWS Production Plan

| Layer | AWS-oriented implementation |
|---|---|
| Scheduling | EventBridge scheduled refreshes |
| Orchestration | Step Functions state machines |
| Source workers | Lambda for light connectors, ECS/Fargate or AWS Batch for crawlers/bulk jobs |
| Storage | S3 raw evidence snapshots, RDS/DynamoDB normalized evidence and workflow state |
| LLM fallback | Amazon Bedrock behind source/field/failure gates |
| Review queue | Dashboard backed by RDS/DynamoDB and SQS-style work queues |
| Monitoring | CloudWatch metrics/alarms for freshness, cost drift, review backlog, false positives |
| Audit | Append-only audit events, evidence hashes, rollback plan before mutation |

This is intentionally implementable by a lean team: deterministic MVP first, then source hardening, review dashboard, audit/rollback, and production AWS scheduling.

## 13. Bonus Point Coverage

| Bonus item | Coverage |
|---|---|
| Working prototype | This notebook plus `scripts/run_best_pipeline.py` |
| Agent workflow diagram | `ARCHITECTURE_DIAGRAM.md`, `AGENT_WORKFLOW_DIAGRAM.md` |
| Cost estimate per 1,000 records | `COST_MODEL.md`, `cost_model_per_1000.csv` |
| Confidence scoring formula | This notebook, `FIELD_RISK_POLICY.md`, candidate confidence fields |
| Human review dashboard | `dashboard/index.html`, `dashboard_v2/index.html` |
| Duplicate detection | `DUPLICATE_MOVEMENT_DETECTION.md`, duplicate candidate outputs |
| Address normalization | `normalize_address`, `FIELD_RISK_POLICY.md` |
| NPI validation | `NPPES_API_SMOKE.md`, NPPES evidence artifacts |
| Practice-location matching | movement/practice diagnostics and affiliation artifacts |
| Provider movement detection | `provider_movement_candidates.csv` |
| Inactive provider detection | `INACTIVE_PROVIDER_DETECTION.md` |
| Change history and audit log | `audit_events.jsonl`, `provider_change_timeline.csv` |
| Safe auto-update rules | `auto_apply_updates.csv`, field thresholds |
| Clear roadmap | `PRODUCTION_READINESS_SCORECARD.md`, `AWS_PRODUCTION_ARCHITECTURE.md` |

## Final Takeaway

The strongest submission is not the one that blindly changes the most records. It is the one HealthLynked can trust in production.

This MVP finds real candidate changes, explains each recommendation, protects risky updates with human review, keeps auto-apply conservative, tracks cost, and shows a practical AWS path for scaling to thousands or millions of records.